# Chapter 2 Computational Lab
## $\sigma$-Algebras and the Structure of Events

This notebook accompanies Chapter 2 of *Probability Theory with Python and AI*.

The central idea of the chapter is that probability first needs a precise **event structure**. A $\sigma$-algebra describes which yes--no questions can be answered from the information available in the model.

### Learning goals

By the end of this lab you should be able to:

1. distinguish an algebra from a $\sigma$-algebra;
2. compute generated $\sigma$-algebras on finite sample spaces;
3. understand how partitions encode observable information;
4. interpret coarse and fine information through inclusion of $\sigma$-algebras;
5. work with inverse images and finite versions of measurability;
6. visualize basic Borel constructions and the Cantor set;
7. compute $\liminf A_n$ and $\limsup A_n$ for finite event sequences;
8. distinguish $\pi$-systems, Dynkin systems and $\sigma$-algebras;
9. see computationally how the $\pi$--$\lambda$ extension principle works on finite spaces;
10. audit AI-generated arguments about measurable events and information.

> **Working principle.** On a finite space, computation can verify closure and generation exactly. On an infinite space, the same code is only an analogy; the mathematical definitions remain essential.


## 0. Setup

All finite sets are represented by Python `frozenset` objects. A finite family of events is represented by a Python `set` of `frozenset` objects.


In [ ]:
import math
import random
from itertools import chain, combinations, product

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def powerset(iterable):
    items = list(iterable)
    return {
        frozenset(c)
        for r in range(len(items) + 1)
        for c in combinations(items, r)
    }


def fmt_set(A):
    if not A:
        return r"\varnothing"
    return r"\{" + ",".join(map(str, sorted(A))) + r"\}"


def fmt_family(F):
    ordered = sorted(F, key=lambda A: (len(A), tuple(sorted(A))))
    return r"\left\{" + r",\ ".join(fmt_set(A) for A in ordered) + r"\right\}"


def complement(A, omega):
    return frozenset(omega - A)


def is_sigma_algebra(family, omega):
    F = set(map(frozenset, family))
    O = frozenset(omega)

    if O not in F:
        return False, "The family does not contain Ω."

    for A in F:
        if complement(A, O) not in F:
            return False, f"Missing complement of {sorted(A)}."

    # On a finite family it is enough to check binary unions.
    for A in F:
        for B in F:
            if A | B not in F:
                return False, f"Missing union of {sorted(A)} and {sorted(B)}."

    return True, "All finite σ-algebra closure checks pass."


def is_pi_system(family):
    P = set(map(frozenset, family))
    for A in P:
        for B in P:
            if A & B not in P:
                return False, (A, B, A & B)
    return True, None


def is_dynkin_system(family, omega):
    D = set(map(frozenset, family))
    O = frozenset(omega)

    if O not in D:
        return False, "Ω is missing."

    for A in D:
        if complement(A, O) not in D:
            return False, f"Complement of {sorted(A)} is missing."

    # In a finite universe it is enough to check unions of pairwise disjoint members.
    members = list(D)
    for A in members:
        for B in members:
            if not (A & B) and (A | B) not in D:
                return False, f"Disjoint union of {sorted(A)} and {sorted(B)} is missing."

    return True, "Dynkin closure checks pass."


def generated_sigma_algebra(omega, generators):
    O = frozenset(omega)
    F = {frozenset(), O}
    F.update(frozenset(g) for g in generators)

    changed = True
    while changed:
        changed = False
        current = list(F)

        for A in current:
            Ac = complement(A, O)
            if Ac not in F:
                F.add(Ac)
                changed = True

        current = list(F)
        for A in current:
            for B in current:
                U = A | B
                if U not in F:
                    F.add(U)
                    changed = True
    return F


def generated_dynkin_system(omega, generators):
    O = frozenset(omega)
    D = {frozenset(g) for g in generators}
    D.add(O)

    changed = True
    while changed:
        changed = False
        current = list(D)

        for A in current:
            Ac = complement(A, O)
            if Ac not in D:
                D.add(Ac)
                changed = True

        current = list(D)
        # Add finite disjoint unions. Iteration gives all finite disjoint unions.
        for A in current:
            for B in current:
                if not (A & B):
                    U = A | B
                    if U not in D:
                        D.add(U)
                        changed = True
    return D


def partition_from_map(mapping):
    blocks = {}
    for w, value in mapping.items():
        blocks.setdefault(value, set()).add(w)
    return [frozenset(v) for v in blocks.values()]


def sigma_from_partition(blocks):
    blocks = list(map(frozenset, blocks))
    result = set()
    for r in range(len(blocks) + 1):
        for subset in combinations(blocks, r):
            U = frozenset().union(*subset) if subset else frozenset()
            result.add(U)
    return result


def inverse_image(mapping, target_set):
    return frozenset(w for w, value in mapping.items() if value in target_set)


def show_box(title, text):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b><br>{text}</div>"
    ))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> The exact finite-set tools are ready.</div>"
))


## 1. Outcomes, events and questions

Let $\Omega$ be the sample space. An event is a subset of $\Omega$ that belongs to the selected event family.

For the die example,

$$
\Omega=\{1,2,3,4,5,6\},\qquad
A=\{2,4,6\},\qquad
B=\{4,5,6\}.
$$

The logical questions “not $A$”, “$A$ and $B$”, and “$A$ or $B$” become $A^c$, $A\cap B$, and $A\cup B$.

The power set $\mathcal P(\Omega)$ contains **every** subset of $\Omega$.


In [ ]:
die_omega = frozenset(range(1, 7))
A = frozenset({2, 4, 6})
B = frozenset({4, 5, 6})

display(Math(r"\Omega=" + fmt_set(die_omega)))
display(Math(r"A^c=" + fmt_set(complement(A, die_omega))))
display(Math(r"A\cap B=" + fmt_set(A & B)))
display(Math(r"A\cup B=" + fmt_set(A | B)))
display(Math(rf"|\mathcal P(\Omega)|=2^{{|\Omega|}}=2^6={len(powerset(die_omega))}"))


## 2. Algebra versus $\sigma$-algebra

An algebra is closed under complements and **finite** unions. A $\sigma$-algebra is closed under complements and **countable** unions.

On a finite sample space, the distinction disappears: every algebra is automatically a $\sigma$-algebra because only finitely many distinct subsets exist.

On infinite spaces the distinction matters. The finite--cofinite family on $\mathbb N$ is the standard example: it is an algebra but not a $\sigma$-algebra.

For finite computational work, the following checker tests the corresponding finite closure conditions.


In [ ]:
omega_text = widgets.Text(value="1,2,3,4", description="Ω")
family_text = widgets.Textarea(
    value="; 1,2; 3,4; 1,2,3,4",
    description="Family",
    layout=widgets.Layout(width="700px", height="90px")
)
sigma_check_output = widgets.Output()


def parse_set(text):
    text = text.strip()
    if not text:
        return frozenset()
    return frozenset(int(x.strip()) for x in text.split(",") if x.strip())


def parse_family(text):
    return {parse_set(part) for part in text.split(";")}


def update_sigma_check(*_):
    with sigma_check_output:
        clear_output(wait=True)
        try:
            O = parse_set(omega_text.value)
            F = parse_family(family_text.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers; separate events by semicolons.**"))
            return

        if any(not A <= O for A in F):
            display(Markdown("**Every event must be a subset of Ω.**"))
            return

        ok, message = is_sigma_algebra(F, O)
        display(Math(r"\mathcal F=" + fmt_family(F)))
        display(Markdown(("**Valid finite σ-algebra.** " if ok else "**Not a σ-algebra.** ") + message))


for c in (omega_text, family_text):
    c.observe(update_sigma_check, names="value")

display(widgets.VBox([omega_text, family_text, sigma_check_output]))
update_sigma_check()


### Closure properties

If $\mathcal F$ is a $\sigma$-algebra, then it automatically contains $\varnothing$ and is closed under intersections, differences and symmetric differences.

For example,

$$
A\setminus B=A\cap B^c,
\qquad
A\triangle B=(A\setminus B)\cup(B\setminus A).
$$

The next cell takes two events already present in a finite $\sigma$-algebra and verifies the derived events.


In [ ]:
closure_omega = frozenset({1,2,3,4})
closure_F = generated_sigma_algebra(closure_omega, [{1,2}, {2,3}])
closure_A = frozenset({1,2})
closure_B = frozenset({2,3})

display(Math(r"\sigma(\{A,B\})=" + fmt_family(closure_F)))
display(Math(r"A\cap B=" + fmt_set(closure_A & closure_B)))
display(Math(r"A\setminus B=" + fmt_set(closure_A - closure_B)))
display(Math(r"A\triangle B=" + fmt_set(closure_A ^ closure_B)))


## 3. Generated $\sigma$-algebras

For a family $\mathcal C\subseteq\mathcal P(\Omega)$, the generated $\sigma$-algebra $\sigma(\mathcal C)$ is the **smallest** $\sigma$-algebra containing $\mathcal C$.

On a finite space we can construct it by repeatedly adding:

- $\varnothing$ and $\Omega$,
- complements,
- unions,

until no new set appears.


In [ ]:
gen_omega = widgets.IntSlider(value=5, min=2, max=8, description="|Ω|")
gen_text = widgets.Text(value="1,2; 2,3", description="Generators")
gen_output = widgets.Output()


def update_generated(*_):
    with gen_output:
        clear_output(wait=True)
        O = frozenset(range(1, gen_omega.value + 1))
        try:
            G = parse_family(gen_text.value)
        except ValueError:
            display(Markdown("**Use semicolon-separated integer sets.**"))
            return

        if any(not A <= O for A in G):
            display(Markdown("**Every generator must be a subset of Ω.**"))
            return

        F = generated_sigma_algebra(O, G)
        display(Math(r"\mathcal C=" + fmt_family(G)))
        display(Math(r"\sigma(\mathcal C)=" + fmt_family(F)))
        display(Markdown(
            f"**Number of events:** {len(F)} out of {2**len(O)} possible subsets."
        ))


for c in (gen_omega, gen_text):
    c.observe(update_generated, names="value")

display(widgets.VBox([widgets.HBox([gen_omega, gen_text]), gen_output]))
update_generated()


### One generating event

If $A\neq\varnothing,\Omega$, then

$$
\sigma(\{A\})
=
\{\varnothing,A,A^c,\Omega\}.
$$

One binary observation can distinguish only two blocks: $A$ and $A^c$.


In [ ]:
one_O = frozenset(range(1, 7))
one_A = frozenset({2,4,6})
one_sigma = generated_sigma_algebra(one_O, [one_A])

display(Math(r"\Omega=" + fmt_set(one_O)))
display(Math(r"A=" + fmt_set(one_A)))
display(Math(r"\sigma(\{A\})=" + fmt_family(one_sigma)))


## 4. Partitions and observable information

A partition $\{C_i:i\in I\}$ consists of non-empty, pairwise disjoint blocks whose union is $\Omega$.

For a finite or countable partition,

$$
\sigma(\{C_i:i\in I\})
=
\left\{
\bigcup_{i\in J}C_i:J\subseteq I
\right\}.
$$

So a partition-generated event must be a union of **whole blocks**. The model cannot distinguish two outcomes that lie in the same block.


In [ ]:
partition_omega = frozenset(range(1, 10))
partition_blocks = [
    frozenset({1,2,3}),
    frozenset({4,5}),
    frozenset({6,7,8,9}),
]
partition_sigma = sigma_from_partition(partition_blocks)

display(Math(r"\Omega=" + fmt_set(partition_omega)))
display(Math(r"\mathcal F=" + fmt_family(partition_sigma)))
display(Markdown(
    f"Three blocks generate **{len(partition_sigma)} = 2^3** observable events."
))

fig, ax = plt.subplots(figsize=(8, 2.6))
starts = [0, 3, 5]
widths = [3, 2, 4]
labels = ["L", "M", "H"]
for start, width, label in zip(starts, widths, labels):
    ax.barh([0], [width], left=[start], alpha=0.25, edgecolor="black")
    ax.text(start + width/2, 0, label, ha="center", va="center", fontsize=16)
ax.set_xlim(0, 9)
ax.set_yticks([])
ax.set_xticks(range(10))
ax.set_title("Information from a three-block partition")
plt.show()


### Interactive partition generator

Enter a label for each outcome. Outcomes with the same label belong to the same observable block.

For example, on $\Omega=\{1,\dots,6\}$ the labels

`odd, even, odd, even, odd, even`

generate exactly the parity information.


In [ ]:
part_n = widgets.IntSlider(value=6, min=2, max=10, description="|Ω|")
part_labels = widgets.Text(
    value="odd,even,odd,even,odd,even",
    description="Labels",
    layout=widgets.Layout(width="650px")
)
part_output = widgets.Output()


def update_partition(*_):
    with part_output:
        clear_output(wait=True)
        n = part_n.value
        labels = [x.strip() for x in part_labels.value.split(",")]
        if len(labels) != n:
            display(Markdown(f"**Provide exactly {n} labels.**"))
            return

        mapping = {i+1: labels[i] for i in range(n)}
        blocks = partition_from_map(mapping)
        F = sigma_from_partition(blocks)

        display(Markdown(f"**Blocks:** `{[sorted(b) for b in blocks]}`"))
        display(Math(r"\sigma(\text{blocks})=" + fmt_family(F)))
        display(Markdown(
            f"Number of distinct observable blocks = **{len(blocks)}**, "
            f"so the generated σ-algebra has **{len(F)} = 2^{len(blocks)}** events."
        ))


for c in (part_n, part_labels):
    c.observe(update_partition, names="value")

display(widgets.VBox([widgets.HBox([part_n, part_labels]), part_output]))
update_partition()


## 5. Countable--cocountable $\sigma$-algebra

On an uncountable sample space $\Omega$, define

$$
\mathcal C
=
\{A\subseteq\Omega:
A\text{ is countable or }A^c\text{ is countable}\}.
$$

This is a $\sigma$-algebra. It is much smaller than $\mathcal P(\Omega)$.

For $\Omega=\mathbb R$:

- $\mathbb Q$ is countable, so $\mathbb Q\in\mathcal C$;
- $\mathbb R\setminus\mathbb Q$ is cocountable, so it also belongs to $\mathcal C$;
- $(0,1)$ and its complement are both uncountable, so $(0,1)\notin\mathcal C$.

This example cannot be faithfully represented by a finite checker, because on a finite space every set is countable. The point of the section is precisely the new behavior that appears on uncountable spaces.


## 6. Borel sets on the real line

The Borel $\sigma$-algebra is

$$
\mathcal B(\mathbb R)
=
\sigma(\{G\subseteq\mathbb R:G\text{ is open}\}).
$$

Several much simpler families generate the same $\sigma$-algebra, including all closed left rays $(-\infty,x]$.

Typical constructions are

$$
(a,b]
=
(-\infty,b]\setminus(-\infty,a],
$$

and

$$
(-\infty,b)
=
\bigcup_{n=1}^{\infty}
(-\infty,b-1/n].
$$

The next picture visualizes the second approximation.


In [ ]:
borel_b = widgets.FloatSlider(value=2.0, min=-2.0, max=4.0, step=0.25, description="b")
borel_N = widgets.IntSlider(value=6, min=1, max=20, description="N")
borel_output = widgets.Output()


def update_borel(*_):
    with borel_output:
        clear_output(wait=True)
        b = borel_b.value
        N = borel_N.value

        display(Math(
            rf"(-\infty,{b:g})\approx"
            rf"\bigcup_{{n=1}}^{{{N}}}(-\infty,{b:g}-1/n]"
        ))

        fig, ax = plt.subplots(figsize=(9, 2.5))
        xmin = b - 4
        xs = np.linspace(xmin, b + 0.5, 600)
        ax.axvline(b, linestyle="--", label="open endpoint b")
        for n in [1, max(2, N//2), N]:
            endpoint = b - 1/n
            ax.plot([xmin, endpoint], [n, n], linewidth=4, alpha=0.55)
            ax.plot(endpoint, n, "o")
        ax.set_xlim(xmin, b + 0.5)
        ax.set_yticks([])
        ax.set_xlabel("real line")
        ax.set_title("Closed left rays increasing toward an open left ray")
        plt.show()


for c in (borel_b, borel_N):
    c.observe(update_borel, names="value")

display(widgets.VBox([widgets.HBox([borel_b, borel_N]), borel_output]))
update_borel()


### Basic Borel sets

Open sets, closed sets, intervals, singletons and countable subsets of $\mathbb R$ are Borel.

For instance,

$$
\{x\}
=
\bigcap_{n=1}^{\infty}
(x-1/n,x+1/n).
$$

The shrinking-interval picture below makes this construction visible.


In [ ]:
singleton_x = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.25, description="x")
singleton_N = widgets.IntSlider(value=6, min=2, max=20, description="N")
single_output = widgets.Output()


def update_singleton(*_):
    with single_output:
        clear_output(wait=True)
        x = singleton_x.value
        N = singleton_N.value

        fig, ax = plt.subplots(figsize=(8, 3))
        for n in range(1, N + 1):
            ax.plot([x - 1/n, x + 1/n], [n, n], linewidth=3)
        ax.axvline(x, linestyle="--")
        ax.set_xlabel("real line")
        ax.set_ylabel("n")
        ax.set_title(rf"Intervals $(x-1/n,x+1/n)$ shrinking to $x={x:g}$")
        plt.show()


for c in (singleton_x, singleton_N):
    c.observe(update_singleton, names="value")

display(widgets.VBox([widgets.HBox([singleton_x, singleton_N]), single_output]))
update_singleton()


## 7. Historical problem: the Cantor set

Start with $C_0=[0,1]$. At each stage remove the open middle third of every remaining interval.

Each $C_n$ is a finite union of closed intervals, hence closed and Borel. The Cantor set is

$$
C
=
\bigcap_{n=0}^{\infty}C_n,
$$

so it is also closed and therefore Borel.

The construction is a useful warning: a Borel set can have highly nontrivial geometry.


In [ ]:
cantor_stage = widgets.IntSlider(value=5, min=0, max=8, description="stage n")
cantor_output = widgets.Output()


def cantor_intervals(stage):
    intervals = [(0.0, 1.0)]
    for _ in range(stage):
        new = []
        for a, b in intervals:
            third = (b - a) / 3
            new.append((a, a + third))
            new.append((b - third, b))
        intervals = new
    return intervals


def update_cantor(*_):
    with cantor_output:
        clear_output(wait=True)
        n = cantor_stage.value

        fig, ax = plt.subplots(figsize=(10, 1.8))
        for a, b in cantor_intervals(n):
            ax.plot([a, b], [0, 0], linewidth=8)
        ax.set_xlim(-0.02, 1.02)
        ax.set_yticks([])
        ax.set_title(rf"Cantor construction: $C_{n}$ has $2^{n}$ intervals")
        plt.show()

        display(Math(
            rf"\text{{number of intervals}}=2^{{{n}}}={2**n},\qquad"
            rf"\text{{length of each}}=3^{{-{n}}}"
        ))


cantor_stage.observe(update_cantor, names="value")
display(widgets.VBox([cantor_stage, cantor_output]))
update_cantor()


## 8. Observations and induced information

Let $X:\Omega\to S$. For $B\subseteq S$, the inverse image is

$$
X^{-1}(B)
=
\{\omega\in\Omega:X(\omega)\in B\}.
$$

Inverse images preserve complements, unions and intersections:

$$
X^{-1}(B^c)
=
\bigl(X^{-1}(B)\bigr)^c,
$$

$$
X^{-1}\left(\bigcup_n B_n\right)
=
\bigcup_n X^{-1}(B_n).
$$

This is the structural reason measurable information can be pulled back from the value space to the sample space.


In [ ]:
obs_mapping = {1: 0, 2: 0, 3: 1, 4: 1, 5: 2, 6: 2}
target_S = frozenset({0,1,2})
target_B1 = frozenset({0,1})
target_B2 = frozenset({1,2})

pre_B1 = inverse_image(obs_mapping, target_B1)
pre_B2 = inverse_image(obs_mapping, target_B2)

display(Math(r"X^{-1}(B_1)=" + fmt_set(pre_B1)))
display(Math(r"X^{-1}(B_2)=" + fmt_set(pre_B2)))
display(Math(
    r"X^{-1}(B_1\cup B_2)="
    + fmt_set(inverse_image(obs_mapping, target_B1 | target_B2))
))
display(Math(
    r"X^{-1}(B_1)\cup X^{-1}(B_2)="
    + fmt_set(pre_B1 | pre_B2)
))


### $\sigma(X)$ on a finite value space

If $S$ is finite and carries its full power set, then

$$
\sigma(X)
=
\{X^{-1}(B):B\subseteq S\}.
$$

Computationally this is just the $\sigma$-algebra generated by the level sets of $X$.


In [ ]:
obs_n = widgets.IntSlider(value=6, min=2, max=10, description="|Ω|")
obs_values = widgets.Text(
    value="0,0,1,1,2,2",
    description="X values",
    layout=widgets.Layout(width="650px")
)
obs_output = widgets.Output()


def update_observation(*_):
    with obs_output:
        clear_output(wait=True)
        n = obs_n.value
        try:
            values = [int(x.strip()) for x in obs_values.value.split(",")]
        except ValueError:
            display(Markdown("**Use comma-separated integer values.**"))
            return

        if len(values) != n:
            display(Markdown(f"**Provide exactly {n} values.**"))
            return

        mapping = {i+1: values[i] for i in range(n)}
        blocks = partition_from_map(mapping)
        F = sigma_from_partition(blocks)

        display(Markdown(f"**Level sets of X:** `{[sorted(b) for b in blocks]}`"))
        display(Math(r"\sigma(X)=" + fmt_family(F)))
        display(Markdown(
            f"$X$ distinguishes **{len(blocks)}** observational classes, "
            f"so $\\sigma(X)$ contains **{len(F)}** events."
        ))


for c in (obs_n, obs_values):
    c.observe(update_observation, names="value")

display(widgets.VBox([widgets.HBox([obs_n, obs_values]), obs_output]))
update_observation()


## 9. Measurability on a finite space

A map

$$
X:(\Omega,\mathcal F)\to(S,\mathcal S)
$$

is measurable when

$$
X^{-1}(B)\in\mathcal F
\qquad
\text{for every }B\in\mathcal S.
$$

Equivalently,

$$
\sigma(X)\subseteq\mathcal F.
$$

On a finite codomain with $\mathcal S=\mathcal P(S)$, it is enough to check the level sets of $X$.


In [ ]:
meas_O = frozenset({1,2,3,4})
meas_F = {
    frozenset(),
    frozenset({1,2}),
    frozenset({3,4}),
    frozenset({1,2,3,4}),
}
meas_X = {1:0, 2:0, 3:1, 4:1}

meas_blocks = partition_from_map(meas_X)
meas_sigma_X = sigma_from_partition(meas_blocks)

display(Math(r"\mathcal F=" + fmt_family(meas_F)))
display(Math(r"\sigma(X)=" + fmt_family(meas_sigma_X)))
display(Markdown(
    "**Measurable:** " + str(meas_sigma_X <= meas_F)
))


### Continuous transformations: a calculus-first reminder

Every continuous $g:\mathbb R\to\mathbb R$ is Borel measurable. Therefore if $X$ is a real-valued measurable observation, then $g(X)$ is measurable.

For example, $g(x)=x^2$ is continuous, and

$$
g^{-1}((-\infty,4])
=
[-2,2].
$$

The statement “continuous functions are Borel measurable” is a standard source of examples; it is **not** the definition of Borel measurability.


## 10. Information loss under transformations

If $Y=g(X)$ for a measurable transformation $g$, then

$$
\sigma(Y)\subseteq\sigma(X).
$$

The inclusion may be strict: a transformation can discard information.

A useful finite example is observing the **sum** of two coordinates instead of the ordered pair.


In [ ]:
pair_omega = frozenset(product([0,1,2], repeat=2))
pair_mapping = {w: w for w in pair_omega}
sum_mapping = {w: sum(w) for w in pair_omega}

pair_blocks = partition_from_map(pair_mapping)
sum_blocks = partition_from_map(sum_mapping)

sigma_pair = sigma_from_partition(pair_blocks)
sigma_sum = sigma_from_partition(sum_blocks)

display(Markdown(f"**Ordered-pair level sets:** {len(pair_blocks)}"))
display(Markdown(f"**Sum level sets:** {len(sum_blocks)}"))
display(Markdown(f"$|\\sigma(X_1,X_2)|={len(sigma_pair)}$"))
display(Markdown(f"$|\\sigma(S)|={len(sigma_sum)}$"))
display(Markdown(f"$\\sigma(S)\\subseteq\\sigma(X_1,X_2)$: **{sigma_sum <= sigma_pair}**"))

sum_classes = {}
for w, s in sum_mapping.items():
    sum_classes.setdefault(s, []).append(w)
display(Markdown("**Outcomes merged by observing only the sum:**"))
for s in sorted(sum_classes):
    display(Markdown(f"- $S={s}$: `{sum_classes[s]}`"))


### Binned observations

If a real-valued observation is recorded only in broad reporting bands, exact-value information is lost.

For example,

$$
C_1=(-\infty,0],
\qquad
C_2=(0,1],
\qquad
C_3=(1,\infty).
$$

The binned variable identifies only which $C_i$ occurred. Its generated $\sigma$-algebra consists of unions of the corresponding level sets.


In [ ]:
binned_values = {
    1: -1.0,
    2: 0.0,
    3: 0.25,
    4: 0.5,
    5: 1.0,
    6: 1.5,
    7: 3.0,
}

def band(x):
    if x <= 0:
        return 1
    if x <= 1:
        return 2
    return 3

Y = {w: band(x) for w, x in binned_values.items()}
sigma_X_finite = sigma_from_partition(partition_from_map(binned_values))
sigma_Y = sigma_from_partition(partition_from_map(Y))

display(Markdown(f"$|\\sigma(X)|={len(sigma_X_finite)}$"))
display(Markdown(f"$|\\sigma(Y)|={len(sigma_Y)}$"))
display(Markdown(f"$\\sigma(Y)\\subseteq\\sigma(X)$: **{sigma_Y <= sigma_X_finite}**"))
display(Markdown("The exact observations $0.25$, $0.5$, and $1.0$ are merged into the same reporting band."))


## 11. Trace and product $\sigma$-algebras

If $(\Omega,\mathcal F)$ is a measurable space and $D\subseteq\Omega$, the trace is

$$
\mathcal F|_D
=
\{A\cap D:A\in\mathcal F\}.
$$

For two measurable spaces,

$$
\mathcal F_1\otimes\mathcal F_2
=
\sigma\{A_1\times A_2:
A_1\in\mathcal F_1,\ A_2\in\mathcal F_2\}.
$$

On finite spaces with full power-set $\sigma$-algebras, the product $\sigma$-algebra is again the full power set of the Cartesian product.


In [ ]:
trace_O = frozenset({1,2,3,4})
trace_F = {
    frozenset(),
    frozenset({1,2}),
    frozenset({3,4}),
    trace_O,
}
D = frozenset({2,3,4})
trace = {A & D for A in trace_F}

display(Math(r"\mathcal F|_D=" + fmt_family(trace)))

O1 = frozenset({"H","T"})
O2 = frozenset({1,2,3})
prod_O = frozenset(product(O1, O2))
full_product = powerset(prod_O)
display(Markdown(
    f"Product space size: **{len(prod_O)}** outcomes; "
    f"full product σ-algebra size: **{len(full_product)} = 2^{len(prod_O)}** events."
))


## 12. Sequences and limit events

For events $A_1,A_2,\ldots$,

$$
\liminf_{n\to\infty}A_n
=
\bigcup_{n=1}^{\infty}
\bigcap_{k=n}^{\infty}A_k,
$$

and

$$
\limsup_{n\to\infty}A_n
=
\bigcap_{n=1}^{\infty}
\bigcup_{k=n}^{\infty}A_k.
$$

Interpretation:

- $\liminf A_n$: the outcome belongs to **all but finitely many** $A_n$;
- $\limsup A_n$: the outcome belongs to **infinitely many** $A_n$.

To make these notions exactly computable, the next panel uses an **eventually periodic sequence**. In that setting membership can be decided from the repeating tail.


In [ ]:
limit_omega = frozenset({1,2,3,4,5})
tail_A = widgets.Text(value="1,2,3", description="Tail A")
tail_B = widgets.Text(value="2,3,4", description="Tail B")
limit_mode = widgets.Dropdown(
    options=[
        ("Alternate A,B,A,B,...", "alternate"),
        ("Eventually constant A,A,A,...", "constant"),
    ],
    value="alternate",
    description="Tail",
)
limit_output = widgets.Output()


def update_limits(*_):
    with limit_output:
        clear_output(wait=True)
        try:
            A = parse_set(tail_A.value)
            B = parse_set(tail_B.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers.**"))
            return

        if not A <= limit_omega or not B <= limit_omega:
            display(Markdown("**Events must be subsets of Ω={1,2,3,4,5}.**"))
            return

        if limit_mode.value == "alternate":
            # In an infinite alternating tail, eventual membership requires membership in both;
            # infinitely-often membership requires membership in at least one.
            linf = A & B
            lsup = A | B
        else:
            linf = A
            lsup = A

        display(Math(r"\liminf A_n=" + fmt_set(linf)))
        display(Math(r"\limsup A_n=" + fmt_set(lsup)))
        display(Markdown(
            "**Check:** " +
            ("$\\liminf A_n\\subseteq\\limsup A_n$." if linf <= lsup else "unexpected failure")
        ))


for c in (tail_A, tail_B, limit_mode):
    c.observe(update_limits, names="value")

display(widgets.VBox([
    widgets.HBox([tail_A, tail_B]),
    limit_mode,
    limit_output,
]))
update_limits()


### Monotone event sequences

If $A_n\uparrow A$, then both $\liminf A_n$ and $\limsup A_n$ equal $A$. Likewise for $A_n\downarrow A$.

The finite picture below shows an increasing sequence whose union stabilizes at $\Omega$.


In [ ]:
mono_O = frozenset(range(1, 8))
increasing_events = [frozenset(range(1, n+1)) for n in range(1, 8)]

for n, A in enumerate(increasing_events, 1):
    display(Math(rf"A_{n}=" + fmt_set(A)))

display(Math(r"\bigcup_{n=1}^{7}A_n=" + fmt_set(frozenset().union(*increasing_events))))


## 13. $\pi$-systems and Dynkin systems

A family $\mathcal P$ is a $\pi$-system if it is closed under finite intersections:

$$
A,B\in\mathcal P
\Longrightarrow
A\cap B\in\mathcal P.
$$

A family $\mathcal D$ is a Dynkin system if:

1. $\Omega\in\mathcal D$;
2. $A\in\mathcal D\Rightarrow A^c\in\mathcal D$;
3. it is closed under countable unions of pairwise disjoint members.

A Dynkin system that is also a $\pi$-system is a $\sigma$-algebra.


In [ ]:
pidyn_omega = widgets.Text(value="1,2,3,4", description="Ω")
pidyn_family = widgets.Textarea(
    value="; 1,2; 3,4; 1,2,3,4",
    description="Family",
    layout=widgets.Layout(width="700px", height="90px")
)
pidyn_output = widgets.Output()


def update_pidyn(*_):
    with pidyn_output:
        clear_output(wait=True)
        try:
            O = parse_set(pidyn_omega.value)
            F = parse_family(pidyn_family.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers; semicolons separate events.**"))
            return

        if any(not A <= O for A in F):
            display(Markdown("**Every event must lie inside Ω.**"))
            return

        pi_ok, pi_fail = is_pi_system(F)
        dyn_ok, dyn_msg = is_dynkin_system(F, O)
        sig_ok, sig_msg = is_sigma_algebra(F, O)

        display(Markdown(f"**π-system:** {pi_ok}"))
        if not pi_ok and pi_fail:
            A, B, C = pi_fail
            display(Markdown(
                f"Counterexample: `{sorted(A)}` ∩ `{sorted(B)}` = `{sorted(C)}` is missing."
            ))
        display(Markdown(f"**Dynkin system:** {dyn_ok}"))
        display(Markdown(f"**σ-algebra:** {sig_ok}"))


for c in (pidyn_omega, pidyn_family):
    c.observe(update_pidyn, names="value")

display(widgets.VBox([pidyn_omega, pidyn_family, pidyn_output]))
update_pidyn()


## 14. Why the $\pi$--$\lambda$ theorem is useful

The recurring extension pattern is

$$
\boxed{
\text{simple generating $\pi$-system}
\Longrightarrow
\text{Dynkin class where the property holds}
\Longrightarrow
\text{whole generated $\sigma$-algebra}
}
$$

A prototype is uniqueness from closed left rays:

$$
P((-\infty,x])=Q((-\infty,x])
\quad\text{for every }x
$$

and

$$
\mathcal D
=
\{B\in\mathcal B(\mathbb R):P(B)=Q(B)\}.
$$

If $\mathcal D$ is a Dynkin system containing the ray $\pi$-system, the $\pi$--$\lambda$ theorem upgrades agreement on the generators to agreement on every Borel set.

The finite experiment below verifies the abstract identity

$$
\lambda(\mathcal P)=\sigma(\mathcal P)
$$

whenever $\mathcal P$ is a $\pi$-system.


In [ ]:
pl_omega = widgets.Text(value="1,2,3,4", description="Ω")
pl_generators = widgets.Text(
    value="1,2; 2; ",
    description="π-family",
    layout=widgets.Layout(width="650px")
)
pl_output = widgets.Output()


def update_pi_lambda(*_):
    with pl_output:
        clear_output(wait=True)
        try:
            O = parse_set(pl_omega.value)
            P = parse_family(pl_generators.value)
        except ValueError:
            display(Markdown("**Use semicolon-separated sets.**"))
            return

        if any(not A <= O for A in P):
            display(Markdown("**Every set must lie inside Ω.**"))
            return

        pi_ok, fail = is_pi_system(P)
        display(Markdown(f"**Input is a π-system:** {pi_ok}"))
        if not pi_ok:
            if fail:
                A, B, C = fail
                display(Markdown(
                    f"Missing intersection `{sorted(C)}` of `{sorted(A)}` and `{sorted(B)}`."
                ))
            return

        lam = generated_dynkin_system(O, P)
        sig = generated_sigma_algebra(O, P)

        display(Math(r"\lambda(\mathcal P)=" + fmt_family(lam)))
        display(Math(r"\sigma(\mathcal P)=" + fmt_family(sig)))
        display(Markdown(f"**Equal:** {lam == sig}"))


for c in (pl_omega, pl_generators):
    c.observe(update_pi_lambda, names="value")

display(widgets.VBox([pl_omega, pl_generators, pl_output]))
update_pi_lambda()


## 15. Information at different resolutions

A $\sigma$-algebra is a mathematical description of **available information**.

If

$$
\mathcal G\subseteq\mathcal F,
$$

then $\mathcal G$ is coarser: it can answer fewer yes--no questions.

On a finite space, a partition gives a useful picture. A finer partition has smaller blocks and therefore a larger generated $\sigma$-algebra.


In [ ]:
info_O = frozenset(range(1, 9))

coarse_blocks = [
    frozenset({1,2,3,4}),
    frozenset({5,6,7,8}),
]
medium_blocks = [
    frozenset({1,2}),
    frozenset({3,4}),
    frozenset({5,6}),
    frozenset({7,8}),
]
fine_blocks = [frozenset({i}) for i in info_O]

F_coarse = sigma_from_partition(coarse_blocks)
F_medium = sigma_from_partition(medium_blocks)
F_fine = sigma_from_partition(fine_blocks)

display(Markdown(f"Coarse information: **{len(F_coarse)}** events"))
display(Markdown(f"Medium information: **{len(F_medium)}** events"))
display(Markdown(f"Fine information: **{len(F_fine)}** events"))
display(Markdown(
    f"$\\mathcal F_{{coarse}}\\subseteq\\mathcal F_{{medium}}\\subseteq\\mathcal F_{{fine}}$: "
    f"**{F_coarse <= F_medium <= F_fine}**"
))


## 16. Finite computational verification

On a finite sample space, repeated closure under complements and unions computes $\sigma(\mathcal C)$ exactly.

This is educational because it makes “smallest $\sigma$-algebra containing the generators” concrete. It is not a practical method for large spaces: the power set has size $2^{|\Omega|}$.


In [ ]:
growth_omega = frozenset(range(1, 8))
candidate_generators = [
    frozenset({1,2,3}),
    frozenset({3,4,5}),
    frozenset({5,6,7}),
]

for r in range(1, len(candidate_generators) + 1):
    G = candidate_generators[:r]
    F = generated_sigma_algebra(growth_omega, G)
    display(Markdown(
        f"Using the first **{r}** generator(s): "
        f"$|\\sigma(\\mathcal C)|={len(F)}$ events."
    ))


## 17. Guided exercise generator

Solve each question on paper before using **Check** or **Reveal**.


In [ ]:
rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Generated σ-algebra", "generated"),
        ("Partition information", "partition"),
        ("Measurability", "measurable"),
        ("π / Dynkin / σ", "pidyn"),
        ("Limit events", "limits"),
    ],
    value="random",
    description="Type",
)
new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
answer_box = widgets.Text(description="Answer")
check_button = widgets.Button(description="Check")
exercise_prompt = widgets.Output()
exercise_feedback = widgets.Output()
exercise_state = {}


def make_ch2_exercise(_=None):
    kind = exercise_kind.value
    if kind == "random":
        kind = rng.choice(["generated", "partition", "measurable", "pidyn", "limits"])

    if kind == "generated":
        O = frozenset(range(1, 5))
        A = frozenset(rng.sample(list(O), 2))
        F = generated_sigma_algebra(O, [A])
        ans = len(F)
        prompt = f"On Ω={{1,2,3,4}}, let A={sorted(A)}. How many events are in σ({{A}})?"
        hint = "For one nontrivial generating event, include ∅, Ω, A and Aᶜ."
        solution = rf"\sigma(\{{A\}})=\{{\varnothing,A,A^c,\Omega\}},\quad |\sigma(\{{A\}})|={ans}."
    elif kind == "partition":
        k = rng.randint(2, 5)
        ans = 2**k
        prompt = f"A finite partition has {k} nonempty blocks. How many events are in the σ-algebra generated by the blocks?"
        hint = "Each event is a union of a chosen subcollection of blocks."
        solution = rf"2^{{{k}}}={ans}."
    elif kind == "measurable":
        ans = "yes"
        prompt = "Suppose σ(X)⊆F. Is X measurable with respect to F? Answer yes or no."
        hint = "Use the theorem relating measurability and generated information."
        solution = r"\sigma(X)\subseteq\mathcal F\Longleftrightarrow X\text{ is measurable.}"
    elif kind == "pidyn":
        ans = "yes"
        prompt = "A family D is both a Dynkin system and a π-system. Must D be a σ-algebra? Answer yes or no."
        hint = "This is the key lemma before the π–λ theorem."
        solution = r"\text{Yes. A Dynkin }\pi\text{-system is a }\sigma\text{-algebra.}"
    else:
        A = frozenset({1,2,3})
        B = frozenset({2,3,4})
        linf = A & B
        ans = ",".join(map(str, sorted(linf)))
        prompt = "An infinite tail alternates A={1,2,3}, B={2,3,4}, A, B, ... . Enter liminf A_n as comma-separated elements."
        hint = "To occur eventually in an alternating tail, an outcome must belong to both A and B."
        solution = r"\liminf A_n=A\cap B=\{2,3\}."

    exercise_state.clear()
    exercise_state.update(answer=str(ans).lower(), hint=hint, solution=solution)

    answer_box.value = ""
    with exercise_prompt:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))
    with exercise_feedback:
        clear_output(wait=True)


def show_ex_hint(_):
    with exercise_feedback:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + exercise_state["hint"]))


def reveal_ex(_):
    with exercise_feedback:
        clear_output(wait=True)
        display(Math(exercise_state["solution"]))


def check_ex(_):
    with exercise_feedback:
        clear_output(wait=True)
        guess = answer_box.value.strip().lower().replace(" ", "")
        target = exercise_state["answer"].replace(" ", "")
        if guess == target:
            display(Markdown("**Correct.**"))
        else:
            display(Markdown("**Not yet.** Recheck the structural definition before recomputing."))


new_button.on_click(make_ch2_exercise)
hint_button.on_click(show_ex_hint)
reveal_button.on_click(reveal_ex)
check_button.on_click(check_ex)

display(widgets.VBox([
    widgets.HBox([exercise_kind, new_button]),
    exercise_prompt,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    exercise_feedback,
]))
make_ch2_exercise()


## 18. AI Audit

A common AI failure in this chapter is to manipulate formulas correctly while using the **wrong closure property** or the **wrong information structure**.

### Audit protocol

When reviewing an AI answer, ask:

1. Has $\Omega$ been specified?
2. Is every claimed event actually a subset of $\Omega$?
3. For a claimed $\sigma$-algebra, were $\Omega$, complements and countable unions checked?
4. Is the argument accidentally using closure under an **uncountable** union?
5. Is $\sigma(\mathcal C)$ being confused with the generator $\mathcal C$ itself?
6. If a map $X$ is involved, are events described as inverse images?
7. If $\sigma(Y)\subseteq\sigma(X)$ is claimed, can every $Y$-event really be determined from $X$?
8. Are $\liminf A_n$ and $\limsup A_n$ interpreted as “eventually” and “infinitely often” in the correct order?
9. Is a Dynkin system being incorrectly treated as a $\sigma$-algebra without an additional $\pi$-system/intersection argument?
10. In a $\pi$--$\lambda$ argument, is the simple generating family actually a $\pi$-system?

### Suggested AI audit prompts

- “Generate the $\sigma$-algebra on $\Omega=\{1,2,3,4\}$ generated by $\{1,2\}$ and verify each closure property.”
- “Give an example of a Dynkin system that is not obviously a $\sigma$-algebra, then check whether it is also a $\pi$-system.”
- “Explain why $\sigma(g(X))\subseteq\sigma(X)$ for measurable $g$, using inverse images.”
- “Explain the $\pi$--$\lambda$ theorem as an extension mechanism and identify the roles of the $\pi$-system and the Dynkin system separately.”
- “Distinguish $\liminf A_n$ from $\limsup A_n$ using membership of one fixed outcome $\omega$.”


## 19. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. A σ-algebra must be closed under:",
        ["Choose...", "only finite unions", "countable unions", "arbitrary unions", "only intersections"],
        "countable unions",
        r"\text{A }\sigma\text{-algebra is closed under countable unions.}",
    ),
    (
        "2. One nontrivial event A generates how many events?",
        ["Choose...", "2", "3", "4", "8"],
        "4",
        r"\sigma(\{A\})=\{\varnothing,A,A^c,\Omega\}.",
    ),
    (
        "3. A partition with 5 blocks generates:",
        ["Choose...", "5", "10", "25", "32"],
        "32",
        r"2^5=32.",
    ),
    (
        "4. If σ(Y)⊆σ(X), then Y carries:",
        ["Choose...", "at least as much information as X", "no more information than X", "exactly the same information always", "no measurable information"],
        "no more information than X",
        r"\sigma(Y)\subseteq\sigma(X)\text{ means }Y\text{ is no finer than }X.",
    ),
    (
        "5. limsup A_n means:",
        ["Choose...", "all A_n occur", "eventually all occur", "infinitely many occur", "none occur"],
        "infinitely many occur",
        r"\limsup A_n=\{\omega:\omega\in A_n\text{ infinitely often}\}.",
    ),
    (
        "6. A Dynkin system that is also a π-system is:",
        ["Choose...", "always a σ-algebra", "never a σ-algebra", "only an algebra", "necessarily the power set"],
        "always a σ-algebra",
        r"\text{Dynkin}+\pi\text{-system}\Longrightarrow\sigma\text{-algebra}.",
    ),
    (
        "7. The Borel σ-algebra on R is generated by:",
        ["Choose...", "only singletons", "closed left rays", "all subsets of R", "only finite intervals with integer endpoints"],
        "closed left rays",
        r"\mathcal B(\mathbb R)=\sigma\{(-\infty,x]:x\in\mathbb R\}.",
    ),
    (
        "8. The π–λ theorem is mainly an:",
        ["Choose...", "integration formula", "extension principle", "counting identity", "limit theorem for numbers"],
        "extension principle",
        r"\text{Verify on generators and extend to the generated }\sigma\text{-algebra.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    d = widgets.Dropdown(options=options, value="Choose...", layout=widgets.Layout(width="280px"))
    quiz_widgets.append(d)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:650px'>{prompt}</div>"),
        d
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)
        score = sum(
            w.value == correct
            for w, (_, _, correct, _) in zip(quiz_widgets, quiz_data)
        )
        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))
        for i, (w, (_, _, correct, explanation)) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if w.value == correct else "✗"
            display(Markdown(f"**{mark} Question {i}:** correct answer = `{correct}`"))
            display(Math(explanation))


grade_button.on_click(grade_quiz)
display(widgets.VBox(quiz_rows + [grade_button, quiz_output]))


## 20. Automatic mathematical verification

These tests verify the finite algorithms and representative identities used throughout the notebook.


In [ ]:
# Extreme σ-algebras
O = frozenset({1,2,3,4})
trivial = {frozenset(), O}
full = powerset(O)
assert is_sigma_algebra(trivial, O)[0]
assert is_sigma_algebra(full, O)[0]

# One-generator theorem
A = frozenset({1,2})
assert generated_sigma_algebra(O, [A]) == {
    frozenset(), A, O-A, O
}

# Partition theorem
blocks = [frozenset({1,2}), frozenset({3}), frozenset({4})]
Fpart = sigma_from_partition(blocks)
assert len(Fpart) == 2**3
assert is_sigma_algebra(Fpart, O)[0]

# Generated σ-algebra is actually a σ-algebra
G = [frozenset({1,2}), frozenset({2,3})]
Fgen = generated_sigma_algebra(O, G)
assert is_sigma_algebra(Fgen, O)[0]
assert all(g in Fgen for g in G)

# Inverse image identities
mapping = {1:0, 2:0, 3:1, 4:2}
S = frozenset({0,1,2})
B1 = frozenset({0,1})
B2 = frozenset({1,2})
assert inverse_image(mapping, S-B1) == O - inverse_image(mapping, B1)
assert inverse_image(mapping, B1 | B2) == (
    inverse_image(mapping, B1) | inverse_image(mapping, B2)
)
assert inverse_image(mapping, B1 & B2) == (
    inverse_image(mapping, B1) & inverse_image(mapping, B2)
)

# σ(X) from level sets
blocks_X = partition_from_map(mapping)
sigma_X = sigma_from_partition(blocks_X)
assert is_sigma_algebra(sigma_X, O)[0]

# Trace σ-algebra
D = frozenset({2,3,4})
trace = {A & D for A in Fpart}
assert is_sigma_algebra(trace, D)[0]

# π–λ theorem, exhaustively over every π-system on a 3-point universe.
O3 = frozenset({1,2,3})
P3 = list(powerset(O3))
all_families = []
for mask in range(1 << len(P3)):
    fam = {P3[i] for i in range(len(P3)) if mask & (1 << i)}
    if is_pi_system(fam)[0]:
        all_families.append(fam)

for P in all_families:
    lam = generated_dynkin_system(O3, P)
    sig = generated_sigma_algebra(O3, P)
    assert lam == sig

display(Markdown(
    f"**All automatic checks passed.** "
    f"The finite π–λ identity was verified for **{len(all_families)}** π-systems "
    f"on a three-point universe."
))


## 21. Chapter map

The computational ideas in this notebook mirror the chapter's theoretical structure:

| Mathematical idea | Computational analogue |
|---|---|
| $\sigma$-algebra | exact finite closure checker |
| $\sigma(\mathcal C)$ | iterative finite closure |
| partition-generated information | unions of observable blocks |
| Borel generation | interval/ray approximations |
| measurable observation | inverse-image membership checks |
| $\sigma(X)$ | level-set partition of $\Omega$ |
| coarse vs fine information | inclusion of finite generated $\sigma$-algebras |
| $\liminf,\limsup$ | eventually / infinitely-often tail logic |
| $\pi$-system | finite intersection checker |
| Dynkin system | complement + disjoint-union checker |
| $\pi$--$\lambda$ theorem | exact finite equality $\lambda(\mathcal P)=\sigma(\mathcal P)$ |

The guiding conceptual message is that a $\sigma$-algebra is not merely a technical collection of sets: it records **which distinctions the model is capable of observing**.
